In [ ]:
import matplotlib.pyplot as plt
from pymoo.util.nds.non_dominated_sorting import NonDominatedSorting
from pymoo.indicators.igd_plus import IGDPlus
from pymoo.indicators.igd import IGD
import numpy as np
import pandas as pd
import sys
import pickle

from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML


year = 2000
run_file_name = r"/rdata/ian/pico/paperRuns/global_run_table.pkl"
global_pf_file_name = r"/rdata/ian/pico/paperRuns/global_pf.pkl"

full_run_tab = pd.read_pickle(run_file_name)
global_pf = pd.read_pickle(global_pf_file_name)

global_pf = global_pf.sort_values(by="yield")

igd_ind = IGDPlus(global_pf.loc[:,("irr_total", "yield")].values)
#igd_ind = IGD(global_pf.loc[:,("irr_total", "yield")].values)

#global_pf.loc[:,("irr_total", "yield")].values



In [ ]:

run = 0

selection_masks = [
    np.all([full_run_tab['algorithm'] == "pinsga2", full_run_tab['run'] == run, full_run_tab['DM_range'] == "20to30"], axis=0),
    np.all([full_run_tab['algorithm'] == "nsga2", full_run_tab['run'] == run], axis=0)]

labels = ["PI-NSGA-II", "NSGA-II"]

run_tabs = []
facecolors = ['none', 'green', 'purple', 'orange']
edgecolors = ['black', 'green', 'purple', 'orange']
markers = ['s', 'o', 'o', 'o']


In [ ]:
for (m, mask) in enumerate(selection_masks):
    run_tabs.append(full_run_tab.loc[mask, ['irr_total', 'yield', 'gen']])



In [ ]:
def filter_dom(run_tab):

    # Perform non-dominated sorting
    nds = NonDominatedSorting()

    minimize_pop = run_tab.copy()
    minimize_pop["yield"] = minimize_pop["yield"] * -1

    fronts = nds.do(minimize_pop.values, only_non_dominated_front=True)

    return run_tab.iloc[fronts,:].copy()
    

In [ ]:

fig, ax = plt.subplots(figsize=(8,6))

# Set up empty scatter plots for each of the runs 
figures = []
for (d, run_tab) in enumerate(run_tabs):

    figures.append(ax.scatter([], [],
                label=labels[d],
                facecolors=facecolors[d],
                edgecolors=edgecolors[d],
                marker=markers[d])) 

line = ax.plot(global_pf["irr_total"], global_pf["yield"],
            label="Near-true optima",
            color="red")

# Set up labels and such 
ax.set_xlabel("Irrigation (mm)")
ax.set_ylabel("Yield (kg/ha)")

ax.set_xlim(min(global_pf["irr_total"]),max(global_pf["irr_total"]))
ax.set_ylim(min(global_pf["yield"]),max(global_pf["yield"]))

ax.legend()

def animate(gen):

    gen += 1
    rt_single_gen = [
        run_tab[run_tab["gen"] == gen] for (t, run_tab) in enumerate(run_tabs)
    ]

    for (d, run_tab) in enumerate(rt_single_gen):
    
        # Get the Pareto front
        pf = filter_dom(run_tab)
    
        # Plot the data (first column is f1, second column is f2)
        x = pf.iloc[:,0]
        y = pf.iloc[:,1]

        figures[d].set_offsets(np.c_[x,y])

    
    #line = ax.plot(global_pf["irr_total"], global_pf["yield"],
    #            label="Near-true optima",
    #            color="red")



ani = matplotlib.animation.FuncAnimation(fig, animate, 
                frames=200, interval=100, repeat=True) 


# Display the animation in the Jupyter Notebook
HTML(ani.to_html5_video())


In [ ]:


for (d, run_tab) in enumerate(run_tabs):

    max_gen = max(run_tab["gen"]) 

    igdp_vals = []

    for gen in range(1,max_gen):

        # Get the Pareto front
        pf = filter_dom(run_tab[run_tab["gen"] == gen])

        igdp_vals.append(igd_ind(pf.loc[:,("irr_total", "yield")].values.astype(int)))

    
        #if labels[d] == "NSGA-II":
        #    print(igd_ind(pf.loc[:,("irr_total", "yield")].values))
        #    
        #    #print(pf.loc[:,("irr_total", "yield")])
    
    plt.plot(igdp_vals, label=labels[d], color=edgecolors[d])




In [ ]:
import matplotlib.pyplot as plt
import matplotlib.animation
import numpy as np

fig, ax = plt.subplots()
x1, y1, x2, y2 = [],[],[],[]
sc1 = ax.scatter(x1,y1, facecolor="red")
sc2 = ax.scatter(x2,y2, facecolor="blue")
plt.xlim(0,10)
plt.ylim(0,10)

def animate(i):
    x1 = np.random.rand(1)*10
    y1 = np.random.rand(1)*10
    
    x2 = np.random.rand(1)*5
    y2 = np.random.rand(1)*5
    
    sc1.set_offsets(np.c_[x1,y1])
    sc2.set_offsets(np.c_[x2,y2])


ani = matplotlib.animation.FuncAnimation(fig, animate, 
                frames=20, interval=100, repeat=True) 


# Display the animation in the Jupyter Notebook
HTML(ani.to_html5_video())